In [ ]:
# libraries
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


In [ ]:
# globals; Phase 1 study comparison only
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'f_compare'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
SEED = CFG['seed']
sns.set_theme(style='whitegrid', palette='colorblind')

completion_path = OUTPUT / 'comparison_complete.json'
completion_path.unlink(missing_ok=True)

In [ ]:
# traceable inputs; completed evaluation workers have already been collected by the controller
manifest = pd.read_csv(INPUT / 'evaluation_manifest.csv')
training = pd.read_csv(INPUT / 'phase1_manifest.csv')
architectures = pd.read_csv(INPUT / 'architectures.csv')
for code in nbformat.read(INPUT / 'runtime.ipynb', 4).cells:
    exec(compile(code.source, 'runtime.ipynb', 'exec'), globals())
frames, histories, reached = [], [], []
for row in manifest.to_dict('records'):
    folder = INPUT / row['directory']
    frames.append(pd.read_csv(folder / 'replicate_scores.csv'))
    history = pd.read_csv(INPUT / 'phase1_runs' / row['run'] / 'history.csv')
    history['model'], history['regime'], history['replicate'] = row['model'], row['regime'], row['replicate']
    histories.append(history)
    first = history.loc[history['val_success_fraction'].ge(CFG['val_success_min'])].head(1)
    reached.append({'model': row['model'], 'regime': row['regime'], 'replicate': row['replicate'], 'censored': first.empty,
                    **{name: np.nan if first.empty else first.iloc[0][name] for name in ['step', 'total_bits', 'total_cell_updates', 'seconds']}})
scores = pd.concat(frames, ignore_index=True)
history = pd.concat(histories, ignore_index=True)
reach = pd.DataFrame(reached)
lookup = architectures[['model', 'regime', 'parameters', 'parameter_target', 'parameter_relative_gap', 'payload_bits_per_edge_round', 'macs_per_cell_update']]
scores = scores.merge(lookup, on=['model', 'regime'], validate='many_to_one')
scores['estimated_macs_per_image'] = scores['macs_per_cell_update'] * scores['cell_updates_per_image']
scores.to_csv(OUTPUT / 'replicate_scores.csv', index=False)
history.to_csv(OUTPUT / 'training_history.csv', index=False)
reach.to_csv(OUTPUT / 'validation_reach.csv', index=False)


In [ ]:
# uncertainty unit is the independently initialized training run, not pixels/cases/channel draws
metrics = ['foreground_rmse', 'alpha_iou', 'detail_ratio_to_flat', 'success_fraction']
groups = ['model', 'family', 'requested_width', 'width', 'message_bits', 'kind', 'capacity_mode', 'reliability',
          'regime', 'split', 'test_scope', 'flip_rate', 'horizon_multiple', 'steps', 'bits_per_image', 'cell_updates_per_image', 'parameters']
groups += [key for key in ['target_id', 'target_role', 'target_source_sha256', 'target_canvas_sha256'] if key in scores]
comparison = paired_bootstrap(scores, groups, metrics, CFG)
comparison.to_csv(OUTPUT / 'comparison.csv', index=False)
scores[['model', 'regime', 'replicate', 'split', 'test_scope', 'flip_rate', 'horizon_multiple', 'steps', 'target_fraction_outside_seed_cone', 'clean_seed_cone_fg_rmse_bound']].to_csv(OUTPUT / 'reachability.csv', index=False)
validation_reach = reach.groupby(['model', 'regime'], sort=False).agg(
    n_replicates=('replicate', 'nunique'), failure_fraction=('censored', 'mean'),
    median_steps_among_successes=('step', 'median'), median_bits_among_successes=('total_bits', 'median')).reset_index()
validation_reach.to_csv(OUTPUT / 'validation_reach_summary.csv', index=False)
display(comparison.query("metric == 'success_fraction' and split == 'test_joint' and test_scope == 'none' and horizon_multiple == 1"))


In [ ]:
# paired contrasts retain costs and parameter counts so a protocol effect is not mistaken for equal computation
pair_keys = ['requested_width', 'capacity_mode', 'regime', 'replicate', 'split', 'test_scope', 'flip_rate', 'horizon_multiple']
measure = ['foreground_rmse', 'success_fraction']
def paired_difference(left, right, keys, label):
    columns = list(dict.fromkeys(keys + ['model', 'parameters', 'bits_per_image', 'cell_updates_per_image', 'estimated_macs_per_image'] + measure))
    paired = left[columns].merge(right[columns], on=keys, suffixes=('_left', '_right'), validate='one_to_one')
    paired['contrast'] = label
    for metric in measure:
        paired[metric + '_difference'] = paired[metric + '_left'] - paired[metric + '_right']
    paired['equal_bits'] = paired['bits_per_image_left'].eq(paired['bits_per_image_right'])
    paired['equal_updates'] = paired['cell_updates_per_image_left'].eq(paired['cell_updates_per_image_right'])
    return paired
binary = scores.query("kind == 'binary'")
det = binary.query("reliability == 'det'")
contrasts = []
keys = pair_keys + ['family', 'message_bits']
# each reliability intervention gets its own paired comparison against the deterministic same-family baseline
for label, group in binary.query("reliability != 'det'").groupby('reliability', sort=False):
    contrasts.append(paired_difference(group, det, keys, label + '_minus_det'))
query = det.query("family == 'query_answer'").copy()
message = det.query("family == 'message'").copy()
contrasts.append(paired_difference(query, message, pair_keys + ['message_bits'], 'query_reply_minus_message_same_payload'))
# equal per-update wire budget: query(b)+reply(b) versus a single outbound message(2b)
query['match_bits'] = 2 * query['message_bits']
message['match_bits'] = message['message_bits']
contrasts.append(paired_difference(query, message, pair_keys + ['match_bits'], 'query_reply_minus_message_double_payload'))
# same one-way rule gets two state updates per query/reply cycle; the extra update cost is explicit
left = query.query("regime == 'updates' and horizon_multiple == 1").copy()
right = message.query("regime == 'updates' and horizon_multiple == 2").copy()
right['horizon_multiple'] = 1
contrasts.append(paired_difference(left, right, pair_keys + ['message_bits'], 'query_reply_minus_two_oneway_updates'))
float_rows = scores.query("kind == 'float'")
contrasts.append(paired_difference(det, float_rows, pair_keys + ['family', 'message_bits'], 'binary_minus_same_shape_float'))
contrasts.append(paired_difference(float_rows.query("family == 'query_answer'"), float_rows.query("family == 'message'"),
                                  pair_keys + ['message_bits'], 'float_query_reply_minus_float_message'))
paired = pd.concat(contrasts, ignore_index=True)
paired.to_csv(OUTPUT / 'paired_contrasts.csv', index=False)
contrast_keys = ['contrast', 'requested_width', 'capacity_mode', 'regime', 'split', 'test_scope', 'flip_rate', 'horizon_multiple',
                 'model_left', 'model_right', 'parameters_left', 'parameters_right', 'equal_bits', 'equal_updates']
paired_bootstrap(paired, contrast_keys, [m + '_difference' for m in measure], CFG).to_csv(OUTPUT / 'paired_contrast_summary.csv', index=False)
# query-only versus reply-only corruption holds the trained model fixed
q = binary.query("family == 'query_answer' and test_scope == 'q'")
r = binary.query("family == 'query_answer' and test_scope == 'r'")
noise_keys = ['model', 'requested_width', 'capacity_mode', 'regime', 'replicate', 'split', 'flip_rate', 'horizon_multiple']
paired_difference(q, r, noise_keys, 'query_noise_minus_reply_noise').to_csv(OUTPUT / 'query_reply_noise_asymmetry.csv', index=False)


In [ ]:
# budget checks compare every binary configuration at the same reference horizon, including the 8-bit control
budget_source = scores.query("test_scope == 'none'")
if not binary.empty:
    budget_source = budget_source.query("kind == 'binary'")
budget = budget_source.groupby(['regime', 'replicate', 'split', 'horizon_multiple']).agg(
    unique_bit_costs=('bits_per_image', 'nunique'), unique_update_costs=('cell_updates_per_image', 'nunique')).reset_index()
budget['passed'] = np.where(budget['regime'].eq('bits'), budget['unique_bit_costs'].eq(1), budget['unique_update_costs'].eq(1))
budget['scope'] = 'binary' if not binary.empty else 'controls'
assert budget['passed'].all()
budget.to_csv(OUTPUT / 'budget_checks.csv', index=False)
# reconstruction pass is reported without treating execution completion as scientific success
interpretation = pd.DataFrame([
    ('scope', 'one transformed cat; independent test composition/angle/color/translation/joint panels; no Phase 2'),
    ('uncertainty', 'CIs resample training replicates; one replicate yields undefined CI, not zero uncertainty'),
    ('protocol', 'same-width message and query/reply MLPs have unequal parameters; inspect counts or the matched profile'),
    ('float control', 'same-shape float messages isolate quantization at equal updates; their 32-bit components are charged and not bit-matched'),
    ('reliability', 'test flips replace training flip rates; learned Bernoulli stochasticity remains active at every test rate'),
    ('success', 'gate combines foreground error, silhouette overlap, and detail relative to an oracle-mask flat-color baseline'),
    ('limits', 'failure under finite training/communication budgets does not prove representational impossibility')
], columns=['item', 'interpretation'])
interpretation.to_csv(OUTPUT / 'interpretation.csv', index=False)
display(budget)
display(interpretation)


In [ ]:
# accessible figures; aggregate cases/channel trials before plotting uncertainty across training replicates
panel = scores.query("kind == 'binary' and reliability == 'det' and split == 'test_joint' and test_scope == 'none' and horizon_multiple == 1 and not extra_payload_control")
if not panel.empty and not CFG.get('reliability_study', False):
    grid = sns.relplot(data=panel, x='message_bits', y='success_fraction', hue='family', style='family', col='requested_width', row='regime',
                       kind='line', markers=True, dashes=True, errorbar=None, height=2.7, aspect=1.2)
    grid.set_axis_labels('bits per message/query/reply', 'fraction passing reconstruction screen')
    grid.figure.savefig(OUTPUT / 'bandwidth_width.png', dpi=170, bbox_inches='tight')
    plt.show()
noise = scores.query("kind == 'binary' and split == 'test_joint' and horizon_multiple == 1 and message_bits == 1 and test_scope in ['none', 'm', 'qr']")
noise = noise.loc[noise['requested_width'].eq(CFG['widths'][0])]
if not noise.empty and not CFG.get('quantization_study', False):
    grid = sns.relplot(data=noise, x='flip_rate', y='foreground_rmse', hue='reliability', style='reliability', col='family', row='regime',
                       kind='line', markers=True, dashes=True, errorbar=None, height=2.7, aspect=1.3)
    grid.set_axis_labels('external per-bit flip probability', 'foreground RMSE')
    grid.figure.savefig(OUTPUT / 'noise_robustness.png', dpi=170, bbox_inches='tight')
    plt.show()
# fixed plotting subset, never selected for good test performance
width = 64 if 64 in CFG['widths'] else CFG['widths'][0]
chosen = architectures.query("kind == 'binary' and reliability == 'det' and requested_width == @width and message_bits == 1 and regime == 'updates'")['model'].tolist()
if CFG.get('reliability_study', False):
    chosen = architectures.query("kind == 'binary' and not reliability.str.contains('flip')", engine='python')['model'].tolist()
if not chosen:
    chosen = architectures.query("kind in ['float', 'continuous'] and regime == 'updates'")['model'].tolist()
fig, axes = plt.subplots(4, len(chosen), figsize=(3 * len(chosen), 10), squeeze=False)
for column, model_name in enumerate(chosen):
    run = manifest.loc[manifest['model'].eq(model_name) & manifest['regime'].eq('updates') & manifest['replicate'].eq(0)].iloc[0]
    examples = torch.load(INPUT / run['directory'] / 'examples.pt', weights_only=True)
    for row, split in enumerate(['test_composition', 'test_angle', 'test_color', 'test_translation']):
        example = next(item for item in examples if item['split'] == split)
        rgba = torch.cat([example['target'], example['generated']], 1).clamp(0, 1)
        axes[row, column].imshow((rgba[..., :3] + 1 - rgba[..., 3:4]).clamp(0, 1))
        axes[row, column].set_title((model_name.split('__')[0] + '\n' + model_name.split('__')[-2] if CFG.get('reliability_study', False) else (model_name.split('__')[0] + '\n' + model_name.split('__')[-1] if CFG.get('conditioning_diagnostic', False) else model_name.split('__')[0] + '\n' + model_name.split('__')[1])) if row == 0 else split, fontsize=10)
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])
fig.suptitle(('Replicate 0, six binary sampling configurations' if CFG.get('reliability_study', False) else CFG.get('target', {}).get('id', 'Rotation, translation and color')) + '\ntarget left / generated right', fontsize=12)
fig.tight_layout()
fig.savefig(OUTPUT / 'transform_examples.png', dpi=170)
plt.show()

# control diagnostic: complete growth/persistence validation history, with the pool transition marked
if CFG.get('control_diagnostic', False) and not CFG.get('conditioning_diagnostic', False):
    labels = architectures[['model', 'regime', 'family', 'kind', 'width']]
    curves = history.merge(labels, on=['model', 'regime'], validate='many_to_one')
    grid = sns.relplot(data=curves, x='step', y='val_loss', hue='family', style='family', col='width',
                       kind='line', markers=True, dashes=True, errorbar=None, height=3.2, aspect=1.3)
    grid.set_axis_labels('optimizer step', 'validation loss: growth + persistence')
    for ax in grid.axes.flat:
        ax.axvline(CFG['fresh_seed_steps'], color='0.4', linestyle=':', linewidth=1)
    grid.figure.savefig(OUTPUT / 'learning_curves.png', dpi=170, bbox_inches='tight')
    plt.show()

# bounded conditioning diagnostic; raw paired responses and small fixed contact sheet
if CFG.get('conditioning_diagnostic', False):
    responses = pd.concat([pd.read_csv(INPUT / row['directory'] / 'cue_response.csv') for row in manifest.to_dict('records')], ignore_index=True)
    responses.to_csv(OUTPUT / 'cue_response.csv', index=False)
    keys = ['model', 'cue_mode', 'replicate', 'domain', 'factor', 'horizon_multiple']
    summary = responses.groupby(keys, sort=False).agg(pairs=('a', 'size'), output_change_mse=('output_change_mse', 'mean'),
                 target_change_mse=('target_change_mse', 'mean'), response_ratio=('response_ratio', 'mean'),
                 cue_match_advantage=('cue_match_advantage', 'mean'), correct_target_mse=('correct_target_mse', 'mean')).reset_index()
    summary.to_csv(OUTPUT / 'cue_response_summary.csv', index=False)
    display(summary)
    curves = history.copy()
    modes = {s['name']: s['cue_mode'] for s in CFG['models']}
    curves['cue_mode'] = curves['model'].map(modes)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    sns.lineplot(data=curves, x='step', y='val_loss', hue='cue_mode', style='cue_mode', markers=True, errorbar=None, ax=ax)
    ax.axvline(CFG['fresh_seed_steps'], color='0.4', linestyle=':', linewidth=1)
    ax.set_ylabel('validation loss: growth + persistence')
    fig.tight_layout()
    fig.savefig(OUTPUT / 'conditioning_learning.png', dpi=170)
    plt.show()
    fig, axes = plt.subplots(4, len(manifest), figsize=(5 * len(manifest), 10), squeeze=False)
    for col, run in enumerate(manifest.to_dict('records')):
        examples = torch.load(INPUT / run['directory'] / 'cue_examples.pt', weights_only=True)
        for row, example in enumerate(examples):
            # columns: target a, output a, target b, output b
            rgba = torch.cat([example['target'][0], example['generated'][0], example['target'][1], example['generated'][1]], 1).clamp(0, 1)
            axes[row, col].imshow((rgba[..., :3] + 1 - rgba[..., 3:4]).clamp(0, 1))
            axes[row, col].set_title(modes[run['model']] + ' / ' + example['domain'] + ' / ' + example['factor'], fontsize=9)
            axes[row, col].axis('off')
    fig.suptitle('Paired cues: target A, output A, target B, output B')
    fig.tight_layout()
    fig.savefig(OUTPUT / 'conditioning_examples.png', dpi=170)
    plt.show()

# quantization study; pair sensitivity and correctness are separate from reconstruction gates
if CFG.get('quantization_study', False):
    responses = pd.concat([pd.read_csv(INPUT / row['directory'] / 'cue_response.csv') for row in manifest.to_dict('records')], ignore_index=True)
    responses.to_csv(OUTPUT / 'cue_response.csv', index=False)
    keys = ['model', 'family', 'kind', 'message_bits', 'replicate', 'domain', 'factor', 'horizon_multiple']
    response_metrics = ['output_change_mse', 'target_change_mse', 'response_ratio', 'cue_match_advantage', 'correct_target_mse']
    response_scores = responses.groupby(keys, sort=False)[response_metrics].mean().reset_index()
    response_scores.to_csv(OUTPUT / 'cue_response_replicates.csv', index=False)
    response_groups = [k for k in keys if k != 'replicate']
    paired_bootstrap(response_scores, response_groups, response_metrics, CFG).to_csv(OUTPUT / 'cue_response_summary.csv', index=False)
    # matched binary-minus-float differences; never treat pairs as training replicates
    pair_keys_cue = ['family', 'message_bits', 'replicate', 'domain', 'factor', 'horizon_multiple']
    cue_pairs = response_scores.query("kind == 'binary'").merge(response_scores.query("kind == 'float'"),
                     on=pair_keys_cue, suffixes=('_binary', '_float'), validate='one_to_one')
    for metric in response_metrics:
        cue_pairs[metric + '_difference'] = cue_pairs[metric + '_binary'] - cue_pairs[metric + '_float']
    cue_pairs.to_csv(OUTPUT / 'cue_response_contrasts.csv', index=False)
    paired_bootstrap(cue_pairs, [k for k in pair_keys_cue if k != 'replicate'],
                     [m + '_difference' for m in response_metrics], CFG).to_csv(OUTPUT / 'cue_response_contrast_summary.csv', index=False)
    quality = scores.query("kind in ['binary', 'float'] and horizon_multiple == 1")
    grid = sns.relplot(data=quality, x='message_bits', y='foreground_rmse', hue='kind', style='family', col='split', col_wrap=3,
                       kind='line', markers=True, dashes=True, errorbar=None, height=2.6, aspect=1.3)
    for ax, split in zip(grid.axes.flat, grid.col_names):
        baseline = scores.query("kind == 'continuous' and horizon_multiple == 1 and split == @split")['foreground_rmse'].mean()
        ax.axhline(baseline, color='0.3', linestyle=':', label='continuous mean')
        ax.legend(fontsize=8)
    grid.set_axis_labels('binary bits / float components per packet', 'foreground RMSE')
    grid.figure.savefig(OUTPUT / 'quantization_quality.png', dpi=170, bbox_inches='tight')
    plt.show()
    panel = response_scores.query("kind in ['binary', 'float'] and horizon_multiple == 1")
    grid = sns.relplot(data=panel, x='message_bits', y='cue_match_advantage', hue='kind', style='family', col='factor', row='domain',
                       kind='line', markers=True, dashes=True, errorbar=None, height=2.5, aspect=1.3)
    grid.set_axis_labels('binary bits / float components per packet', 'swapped minus correctly paired target MSE')
    grid.figure.savefig(OUTPUT / 'quantization_cue_response.png', dpi=170, bbox_inches='tight')
    plt.show()
    # every validation record retained; figures show means, interval tables use training replicates
    labels = architectures[['model', 'regime', 'family', 'kind', 'message_bits']]
    curves = history.merge(labels, on=['model', 'regime'], validate='many_to_one')
    curves['configuration'] = curves['family'] + ' / ' + curves['kind']
    grid = sns.relplot(data=curves, x='step', y='val_loss', hue='configuration', style='configuration', col='message_bits',
                       kind='line', dashes=True, errorbar=None, height=2.7, aspect=1.3)
    for ax in grid.axes.flat:
        ax.axvline(CFG['fresh_seed_steps'], color='0.4', linestyle=':', linewidth=1)
    grid.figure.savefig(OUTPUT / 'quantization_learning.png', dpi=170, bbox_inches='tight')
    plt.show()
    display(pd.read_csv(OUTPUT / 'cue_response_contrast_summary.csv'))

# bounded reliability study; average pairs/noise trials inside each training replicate
if CFG.get('reliability_study', False):
    responses = pd.concat([pd.read_csv(INPUT / row['directory'] / 'cue_response.csv') for row in manifest.to_dict('records')], ignore_index=True)
    responses.to_csv(OUTPUT / 'cue_response.csv', index=False)
    keys = ['model', 'family', 'kind', 'message_bits', 'reliability', 'replicate', 'domain', 'factor', 'horizon_multiple']
    cue_metrics = ['response_ratio', 'cue_match_advantage', 'correct_target_mse', 'output_change_mse', 'target_change_mse']
    cue_scores = responses.groupby(keys, sort=False)[cue_metrics].mean().reset_index()
    cue_scores.to_csv(OUTPUT / 'cue_response_replicates.csv', index=False)
    paired_bootstrap(cue_scores, [k for k in keys if k != 'replicate'], cue_metrics, CFG).to_csv(OUTPUT / 'cue_response_summary.csv', index=False)
    # primary stress comparison: both packet directions flipped for QR; q/r separately retained in CSVs
    panel = scores.query("kind == 'binary' and horizon_multiple == 1 and split in ['train', 'test_composition', 'test_angle'] and test_scope in ['none', 'm', 'qr']")
    grid = sns.relplot(data=panel, x='flip_rate', y='foreground_rmse', hue='reliability', style='reliability', row='family', col='split',
                       kind='line', markers=True, dashes=True, errorbar=None, height=2.7, aspect=1.3, facet_kws={'margin_titles': True})
    grid.set_axis_labels('external per-bit flip probability', 'foreground RMSE')
    grid.figure.subplots_adjust(hspace=0.35, wspace=0.2)
    grid.figure.savefig(OUTPUT / 'reliability_stress.png', dpi=170, bbox_inches='tight')
    plt.show()
    panel = cue_scores.query("kind == 'binary' and horizon_multiple == 1")
    grid = sns.catplot(data=panel, x='reliability', y='cue_match_advantage', hue='family', col='factor', row='domain',
                       kind='point', errorbar=None, markers=['o', 's'], linestyles=['', ''], height=2.8, aspect=1.6)
    for ax in grid.axes.flat:
        ax.tick_params(axis='x', rotation=45)
    grid.set_axis_labels('training channel configuration', 'swapped minus correctly paired target MSE')
    grid.figure.subplots_adjust(hspace=0.6)
    grid.figure.savefig(OUTPUT / 'reliability_cue_response.png', dpi=170, bbox_inches='tight')
    plt.show()
    labels = architectures[['model', 'regime', 'family', 'reliability', 'kind']]
    curves = history.merge(labels, on=['model', 'regime'], validate='many_to_one')
    curves['configuration'] = curves['kind'] + ' / ' + curves['reliability']
    grid = sns.relplot(data=curves, x='step', y='val_loss', hue='configuration', style='configuration', col='family',
                       kind='line', dashes=True, errorbar=None, height=3, aspect=1.3)
    for ax in grid.axes.flat:
        ax.axvline(CFG['fresh_seed_steps'], color='0.4', linestyle=':', linewidth=1)
    grid.set_axis_labels('optimizer step', 'validation loss under training channel')
    grid.figure.savefig(OUTPUT / 'reliability_learning.png', dpi=170, bbox_inches='tight')
    plt.show()

atomic_json({'execution_complete': True, 'evaluation_runs': len(manifest), 'comparison_rows': len(comparison),
             'updated_at': pd.Timestamp.now(tz='UTC').isoformat()}, completion_path)